# VGG16 on CIFAR-10

这个 Notebook 从 `CIFAR-10` 数据集加载开始，完整展示 `Resize(224x224) -> VGG16` 的训练与结构分析流程。

内容包括：
- 数据集预处理与可视化
- `DataLoader` 构建
- VGG16 模型实现
- 卷积块逐层解读
- 参数量与特征图尺寸分析
- 训练、验证与预测展示
- VGG16 和 AlexNet 的结构差异说明

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# 数学工具主要用于可视化排版等辅助逻辑
import math
# dataclass 用于统一管理实验配置
from dataclasses import dataclass

# matplotlib 负责图像与曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# torchvision 提供常见数据集和图像预处理工具
from torchvision import datasets, transforms

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，方便复现
torch.manual_seed(42)

# 优先使用 GPU，没有则使用 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据下载目录
    data_root: str = './data'
    # 将 CIFAR-10 从 32x32 拉伸到 224x224
    image_size: int = 224
    # 每个 batch 的样本数
    batch_size: int = 64
    # DataLoader 并行加载进程数
    num_workers: int = 2
    # 学习率
    lr: float = 1e-3
    # 训练轮数
    epochs: int = 5


cfg = Config()
cfg

## 2. 加载 CIFAR-10 并拉伸到 224x224

VGG16 通常配合较大尺寸输入一起讨论。这里把 `CIFAR-10` 图片从 `32x32` 拉伸到 `224x224`，便于完整观察 VGG16 中多层卷积和多次池化的空间压缩过程。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# 训练集做尺寸拉伸、随机翻转、张量化和归一化
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 测试集不做随机增强，保持评估稳定
test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 本地不存在数据时自动下载
train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=test_transform,
)

# 类别名称
classes = train_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于显示更自然的图像
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 可视化若干样本，确认输入图像已经被拉伸到 224x224
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), range(8)):
    image, label = train_dataset[idx]
    image = denormalize(image, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 samples resized to 224x224', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 训练集打乱顺序，帮助模型减少对样本顺序的依赖
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    # 使用 GPU 时通常开启 pin_memory 会更高效
    pin_memory=torch.cuda.is_available(),
)

# 测试集保持固定顺序即可
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 检查一个 batch 的维度
images, labels = next(iter(train_loader))
print('batch image shape:', images.shape)
print('batch label shape:', labels.shape)

## 4. VGG16 实现

VGG16 的关键思想是：
- 使用连续的 `3x3` 小卷积核堆叠代替更大的卷积核
- 在每个卷积块后用池化层逐步压缩空间尺寸
- 通过增加网络深度来增强表示能力

下面实现一个适配 `CIFAR-10` 十分类任务的 VGG16。

In [ ]:
class VGG16CIFAR10(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        # features 部分由 5 个卷积块组成，每个卷积块后接一次池化
        self.features = nn.Sequential(
            # Block 1: 从 3 通道映射到 64 通道
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 通道数提升到 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 三层 3x3 卷积堆叠
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4: 通道提升到 512
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5: 最后一个高维卷积块
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 将空间尺寸统一整理为 7x7，保持和经典 VGG 头部兼容的思路
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        # 分类头将高维特征映射为 10 个类别
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        # 逐层提取卷积特征
        x = self.features(x)
        # 整理空间尺寸
        x = self.avgpool(x)
        # 展平成二维张量供全连接层使用
        x = torch.flatten(x, 1)
        # 输出每个类别的 logits
        x = self.classifier(x)
        return x


model = VGG16CIFAR10().to(device)
model

## 5. 逐层尺寸变化分析

VGG16 的结构比 AlexNet 更深，但卷积核更小。下面打印每层输出尺寸，观察空间分辨率如何逐步减半、通道数如何逐步增加。

In [ ]:
def inspect_feature_shapes(model, input_shape=(1, 3, 224, 224)):
    # 构造一个假的输入，仅用于尺寸分析
    x = torch.randn(input_shape)
    print(f'input: {tuple(x.shape)}')

    for idx, layer in enumerate(model.features):
        # 每经过一层打印一次 shape，便于看清 VGG16 的层级结构
        x = layer(x)
        print(f'features[{idx}] {layer.__class__.__name__:<12} -> {tuple(x.shape)}')

    x = model.avgpool(x)
    print(f'avgpool           AdaptiveAvgPool2d -> {tuple(x.shape)}')

    x = torch.flatten(x, 1)
    print(f'flatten                         -> {tuple(x.shape)}')

    for idx, layer in enumerate(model.classifier):
        x = layer(x)
        print(f'classifier[{idx}] {layer.__class__.__name__:<12} -> {tuple(x.shape)}')


inspect_feature_shapes(model.cpu())
model = model.to(device)

### 卷积块解读

1. Block 1 和 Block 2
   - 主要学习低层视觉模式，例如边缘、颜色过渡、简单纹理。
   - 每个卷积后接 `ReLU`，帮助网络学习非线性表达。

2. Block 3
   - 开始进入更复杂的中层结构组合。
   - 三层 `3x3` 卷积连续堆叠，可以用更多非线性变换逼近更复杂模式。

3. Block 4 和 Block 5
   - 通道数维持在 512，侧重提取更抽象、更语义化的高级特征。
   - 对分类任务来说，这些层更接近“物体是什么”而不是“局部纹理长什么样”。

4. 分类头
   - 与经典 VGG 一样，后部全连接层参数量很大。
   - 这也是 VGG16 表达能力强但参数开销高的重要原因。

## 6. 参数量统计

In [ ]:
def count_parameters(model):
    # 只统计参与训练的参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total_params = count_parameters(model)
print(f'Trainable parameters: {total_params:,}')

VGG16 的参数量通常显著高于 AlexNet，尤其是后部大规模全连接层会带来明显的显存和计算开销。这也是后续很多模型开始逐步减少全连接层依赖、转向更高参数效率结构的原因之一。

## 7. 训练与验证函数

In [ ]:
# 多分类任务常用交叉熵损失
criterion = nn.CrossEntropyLoss()
# 使用 Adam，便于快速开始实验
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 进入训练模式
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 将数据移动到对应设备
        images = images.to(device)
        labels = labels.to(device)

        # 清空上一轮梯度
        optimizer.zero_grad()
        # 前向传播
        outputs = model(images)
        # 计算损失
        loss = criterion(outputs, labels)
        # 反向传播
        loss.backward()
        # 更新参数
        optimizer.step()

        # 统计当前 epoch 的累计损失和正确数
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    # 进入评估模式
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 评估时同样需要把数据放到目标设备
        images = images.to(device)
        labels = labels.to(device)

        # no_grad 环境下只做前向传播，不跟踪梯度
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

## 8. 训练主循环

由于 `224x224` 输入和 VGG16 本身都比较重，这部分代码在 CPU 上会比较慢。默认只写出完整训练流程，不主动执行任何额外命令。

In [ ]:
# 记录训练和验证指标，便于后面画曲线
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

# 每轮先训练，再评估
for epoch in range(cfg.epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    # 保存本轮结果
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

In [ ]:
# 绘制损失曲线和准确率曲线
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='train loss')
axes[0].plot(epochs, history['val_loss'], label='val loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='train acc')
axes[1].plot(epochs, history['val_acc'], label='val acc')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, num_images=8):
    # 切换到评估模式，保证推理行为稳定
    model.eval()
    # 取一个 batch 进行可视化
    images, labels = next(iter(dataloader))
    images = images.to(device)
    labels = labels.to(device)

    # 前向传播，获取预测类别
    logits = model(images)
    preds = logits.argmax(dim=1)

    # 以两行形式展示若干预测结果
    fig, axes = plt.subplots(2, math.ceil(num_images / 2), figsize=(16, 6))
    axes = axes.flatten()

    for i in range(num_images):
        # 反归一化后再显示图像
        image = denormalize(images[i].cpu(), cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
        axes[i].imshow(image)
        axes[i].set_title(f'true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}')
        axes[i].axis('off')

    for i in range(num_images, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, classes, device)

## 10. VGG16 相比 AlexNet 的关键差异

### 1. 卷积核设计
AlexNet 前面使用较大的 `11x11` 卷积核，而 VGG16 坚持使用连续 `3x3` 小卷积核堆叠。

### 2. 网络深度
VGG16 更深，层数更多，因此能够构建更复杂的分层特征表达。

### 3. 结构规律性
VGG16 的卷积块结构更整齐，更适合做模块化分析，也更容易作为后续很多 CNN 设计的参考模板。

### 4. 参数与计算成本
VGG16 的参数量和计算成本都更高，尤其是在 `224x224` 输入下更明显。因此它很适合做结构教学，但在实际部署中通常不是最轻量的选择。

## 11. 可继续扩展的方向

- 增加混淆矩阵分析类别误判分布
- 可视化卷积块输出特征图
- 对比 AlexNet 和 VGG16 的参数量与训练速度
- 对比更现代的 ResNet 结构为什么更深但更容易训练